In [523]:
import pandas as pd


In [524]:
population = pd.read_csv(r"C:/Downloads/new_data/Dữ liệu dân số lịch sử.csv")
eu_regions = pd.read_csv(r"C:/Downloads/new_data/eu_country.csv")
population.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Zone de référence,MEASURE,Mesure,UNIT_MEASURE,Unité de measure,...,TIME_PERIOD,Période temporelle,OBS_VALUE,Valeur d'observation,OBS_STATUS,Statut d'observation,UNIT_MULT,Multiplicateur d'unité,DECIMALS,Décimales
0,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Données démographiques historiques,I,AUT,Autriche,POP,Population,PS,Personnes,...,1990,NaN,7677850.0,NaN,A,Valeur normale,0,Units,0,Zero
1,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Données démographiques historiques,I,AUT,Autriche,POP,Population,PS,Personnes,...,1991,NaN,7754891.0,NaN,A,Valeur normale,0,Units,0,Zero
2,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Données démographiques historiques,I,AUT,Autriche,POP,Population,PS,Personnes,...,1992,NaN,7840709.0,NaN,A,Valeur normale,0,Units,0,Zero
3,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Données démographiques historiques,I,AUT,Autriche,POP,Population,PS,Personnes,...,1993,NaN,7905632.0,NaN,A,Valeur normale,0,Units,0,Zero
4,DATAFLOW,OECD.ELS.SAE:DSD_POPULATION@DF_POP_HIST(1.0),Données démographiques historiques,I,AUT,Autriche,POP,Population,PS,Personnes,...,1994,NaN,7936118.0,NaN,A,Valeur normale,0,Units,0,Zero


In [525]:
population.columns

Index(['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'REF_AREA',
       'Zone de référence', 'MEASURE', 'Mesure', 'UNIT_MEASURE',
       'Unité de measure', 'SEX', 'Sexe', 'AGE', 'Âge', 'TIME_HORIZ',
       'Horizon temporel', 'TIME_PERIOD', 'Période temporelle', 'OBS_VALUE',
       'Valeur d'observation', 'OBS_STATUS', 'Statut d'observation',
       'UNIT_MULT', 'Multiplicateur d'unité', 'DECIMALS', 'Décimales'],
      dtype='object')

In [526]:
cols_keep = [
    "TIME_PERIOD",          
    "REF_AREA",             
    "Zone de référence",          
    'AGE',             
    "OBS_VALUE",             
    "UNIT_MULT",
    "UNIT_MEASURE",
]


In [527]:
population[cols_keep]

,TIME_PERIOD,REF_AREA,Zone de référence,AGE,OBS_VALUE,UNIT_MULT,UNIT_MEASURE
0,1990,AUT,Autriche,_T,7.677850e+06,0,PS
1,1991,AUT,Autriche,_T,7.754891e+06,0,PS
2,1992,AUT,Autriche,_T,7.840709e+06,0,PS
3,1993,AUT,Autriche,_T,7.905632e+06,0,PS
4,1994,AUT,Autriche,_T,7.936118e+06,0,PS
...,...,...,...,...,...,...,...
7835,2023,ZAF,Afrique du Sud,Y_LT20,2.258706e+07,0,PS
7836,2023,W,Monde,_T,8.091730e+09,0,PS
7837,2023,W,Monde,Y_LT20,2.667780e+09,0,PS
7838,2023,W,Monde,Y_GE65,8.089161e+08,0,PS


In [528]:
# 1. Lọc các cột bạn đã chọn
test = population[cols_keep]

# 2. Đổi tên cột cho gọn & chuẩn
test = test.rename(columns={
    "TIME_PERIOD": "Year",
    "Zone de référence": "Country",
    "AGE" : "Age",
    "OBS_VALUE": "Value_population",
    "UNIT_MULT": "Unit_mult",
    "UNIT_MEASURE" : "Unit_measure"
})



In [529]:
eu_country = test["REF_AREA"].isin(eu_regions["REF_AREA"])
test = test[eu_country]
test

,Year,REF_AREA,Country,Age,Value_population,Unit_mult,Unit_measure
0,1990,AUT,Autriche,_T,7677850.0,0,PS
1,1991,AUT,Autriche,_T,7754891.0,0,PS
2,1992,AUT,Autriche,_T,7840709.0,0,PS
3,1993,AUT,Autriche,_T,7905632.0,0,PS
4,1994,AUT,Autriche,_T,7936118.0,0,PS
...,...,...,...,...,...,...,...
7799,2023,HRV,Croatie,_T,3859686.0,0,PS
7816,2023,ROU,Roumanie,Y_LT20,4147597.0,0,PS
7817,2023,ROU,Roumanie,Y20T64,11129744.0,0,PS
7818,2023,ROU,Roumanie,Y_GE65,3787874.0,0,PS


In [530]:
test["Age"].unique()

array(['_T', 'Y20T64', 'Y_GE65', 'Y_LT20'], dtype=object)

In [531]:
age_map = {"Y_LT20" : "20-","Y20T64" : "20-64","Y_GE65" : "65+","_T":"Total"}
test["Age"] = test["Age"].replace(age_map)

In [532]:
test.groupby("Year")[["Country","Age","Value_population"]].nunique()

,Country,Age,Value_population
Year,,,
1990,28,4,112
1991,28,4,112
1992,28,4,112
1993,28,4,112
1994,28,4,112
1995,28,4,112
1996,28,4,112
1997,28,4,112
1998,28,4,112


In [533]:
test= test.dropna(subset = ["Value_population"])

In [534]:
test["Unit_measure"].unique()

array(['PS'], dtype=object)

In [535]:
total_df = (
    test[test["Age"] == "Total"][["Year","Country","Value_population"]]
    .rename(columns= {"Value_population" : "Total_population"})
)
total_df

,Year,Country,Total_population
0,1990,Autriche,7677850.0
1,1991,Autriche,7754891.0
2,1992,Autriche,7840709.0
3,1993,Autriche,7905632.0
4,1994,Autriche,7936118.0
...,...,...,...
7753,2023,Suisse,8888818.0
7760,2023,Royaume-Uni,68265209.0
7791,2023,Bulgarie,6446595.5
7799,2023,Croatie,3859686.0


In [536]:
age_df = test[test["Age"] != "Total"].copy()
test = test[test["Age"] != "Total"]


In [537]:
age_df = age_df.merge(
    total_df,
    on=["Country", "Year"],
    how="left"
)

In [538]:
age_df["percent"] = (
    age_df["Value_population"] /
    age_df["Total_population"]
) * 100


In [539]:
age_df.groupby(["Country", "Year"])["percent"].sum()


Country    Year
Allemagne  1990    100.0
           1991    100.0
           1992    100.0
           1993    100.0
           1994    100.0
                   ...  
Suède      2020    100.0
           2021    100.0
           2022    100.0
           2023    100.0
           2024    100.0
Name: percent, Length: 980, dtype: float64

In [540]:
age_df.pop("Total_population")
age_df.pop("Unit_measure")

percent = age_df.pop("percent")
age_df.insert(loc = 4,column= "% Population",value = percent.round(2) )
age_df

,Year,REF_AREA,Country,Age,% Population,Value_population,Unit_mult
0,1990,BEL,Belgique,20-64,60.34,6014461.0,0
1,1991,BEL,Belgique,20-64,60.33,6035618.0,0
2,1992,BEL,Belgique,20-64,60.31,6057927.0,0
3,1993,BEL,Belgique,20-64,60.28,6078623.0,0
4,1994,BEL,Belgique,20-64,60.21,6090317.0,0
...,...,...,...,...,...,...,...
2935,2023,HRV,Croatie,20-64,57.98,2237941.0,0
2936,2023,HRV,Croatie,20-,19.15,738964.0,0
2937,2023,ROU,Roumanie,20-,21.75,4147597.0,0
2938,2023,ROU,Roumanie,20-64,58.38,11129744.0,0


In [542]:
age_df.reset_index(drop = True,inplace= True)
age_df

,Year,REF_AREA,Country,Age,% Population,Value_population,Unit_mult
0,1990,BEL,Belgique,20-64,60.34,6014461.0,0
1,1991,BEL,Belgique,20-64,60.33,6035618.0,0
2,1992,BEL,Belgique,20-64,60.31,6057927.0,0
3,1993,BEL,Belgique,20-64,60.28,6078623.0,0
4,1994,BEL,Belgique,20-64,60.21,6090317.0,0
...,...,...,...,...,...,...,...
2935,2023,HRV,Croatie,20-64,57.98,2237941.0,0
2936,2023,HRV,Croatie,20-,19.15,738964.0,0
2937,2023,ROU,Roumanie,20-,21.75,4147597.0,0
2938,2023,ROU,Roumanie,20-64,58.38,11129744.0,0


In [546]:
age_df.groupby(["Year","Country"])["Age"].size()

Year  Country            
1990  Allemagne              3
      Autriche               3
      Belgique               3
      Bulgarie               3
      Croatie                3
                            ..
2024  Royaume-Uni            3
      République slovaque    3
      Slovénie               3
      Suisse                 3
      Suède                  3
Name: Age, Length: 980, dtype: int64

In [543]:
age_df.to_csv("C:/Downloads/data_clean/population_age.csv",index = False)

